# Laboratorio 4: Análisis de Datos GeoEspaciales

CC3084 – Data Science

Iris Ayala - Anggie Quezada - Jonathan Diaz

Análisis de la proliferación de cianobacteria en los lagos de Atitlán y Amatitlán usando imágenes Sentinel-2.

## Librerías

In [1]:
import os
import openeo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from datetime import date, timedelta
from sentinelhub import SHConfig, SentinelHubRequest, DataCollection, MimeType, CRS, BBox, bbox_to_dimensions

## 1. Conexión con la API de Sentinel-2

In [4]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=BKWR-BPKY 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


## 2. Obtención de los datos raster

### Coordenadas de los lagos

In [5]:
lago_atitlan = {
    "west": -91.326256,
    "east": -91.07151,
    "south": 14.5948,
    "north": 14.750979
}

lago_amatitlan = {
    "west": -90.638065,
    "east": -90.512924,
    "south": 14.412347,
    "north": 14.493799
}

### Fechas oficiales por lago

In [6]:
fechas_atitlan = [
    "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17", "2025-11-21",
    "2025-12-29", "2026-02-12", "2026-03-24", "2026-04-13", "2026-04-28", "2026-07-22"
]

fechas_amatitlan = [
    "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24", "2026-01-08",
    "2026-02-02", "2026-02-07", "2026-03-29", "2026-04-13", "2026-04-28", "2026-06-19"
]

### Descarga de bandas

Se descargan únicamente las bandas B03, B04 y B08, necesarias para calcular NDVI y NDWI.

In [9]:
def descargar_bandas(bbox, fecha, ruta_salida):
    fecha_fin = (date.fromisoformat(fecha) + timedelta(days=1)).isoformat()
    cubo = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[fecha, fecha_fin],
        bands=["B03", "B04", "B08"]
    )
    cubo.download(ruta_salida, format="GTIFF")

In [ ]:
os.makedirs("../data/GIS/atitlan", exist_ok=True)
os.makedirs("../data/GIS/amatitlan", exist_ok=True)

for fecha in fechas_atitlan:
    ruta = f"../data/GIS/atitlan/{fecha}.tif"
    descargar_bandas(lago_atitlan, fecha, ruta)

for fecha in fechas_amatitlan:
    ruta = f"../data/GIS/amatitlan/{fecha}.tif"
    descargar_bandas(lago_amatitlan, fecha, ruta)

### Verificación de una imagen descargada

In [ ]:
with rasterio.open(f"../data/GIS/atitlan/{fechas_atitlan[0]}.tif") as src:
    bandas = src.read()
    print("Número de bandas:", src.count)
    print("Dimensiones:", src.width, src.height)